In [0]:
from pyspark.sql.functions import count, min, max, col

# 1. Read from the Silver Table
silver_df = spark.read.table("f1_project.silver.cleaned_sessions")

# 2. Create an Insight: "Total Races per Country for 2026"
# This shows how many meaningful events (Qualifying/Race) happen per location
gold_df = (silver_df
    .groupBy("country_name", "location")
    .agg(
        count("session_id").alias("total_sessions"),
        min("start_time").alias("event_start_date"),
        max("end_time").alias("event_end_date")
    )
    .orderBy("event_start_date")
)

# 3. Save to Gold Table
(gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("f1_project.gold.race_calendar_summary"))

print("Gold Layer Created! Your dashboard-ready data is live.")
display(spark.table("f1_project.gold.race_calendar_summary"))